# 状態空間モデル

この notebook では、見えている観測の裏に直接は見えない内部状態を置くと、なぜ未来予測や解釈が整理しやすくなるのかを学びます。

## 観測だけでは足りないとき、状態を仮定する

状態空間モデルの発想は単純です。見えている値がノイジーだったり欠けていたりするなら、その裏にもっと滑らかに進む『状態』を置いてしまおう、という考え方です。最初は、状態を「外からは見えないが、裏で進んでいる作業メモ」だと思えば十分です。


この notebook では `position` と `velocity` を観測側に出しつつ、内部では潜在状態 `z` を遷移させます。大事なのは数式を暗記することではなく、『内部で何が進んでいるか』と『外から何が見えているか』を分けると何が読みやすくなるかを掴むことです。


## 状態と観測を分けると何が助かるか

観測は外から見える量、状態はその背後で連続的に動いている内部の要約です。観測がぶれても、状態がなめらかに追えていれば長い予測は安定しやすくなります。ここでは線形の最小例で、その分業をそのまま見ます。言い換えると、「目に見える数値は揺れていても、その裏の進み方はもう少し整っている」と考えるわけです。


## 見るべきポイント

`z_next` が遷移式どおり動くか、`decode` が状態をどう観測へ戻すか、ロールアウトを伸ばしたときにどこから誤差が増えるかを追います。ここで `z_next` は『1歩先へ進んだ内部メモ』、`decode` は『そのメモを見える量へ訳す段階』だと思えば十分です。


線形 SSM は入口にすぎませんが、状態と観測を分ける見方は非線形モデルやカルマンフィルタ系にもそのまま残ります。まずはこの小さい系で『何を状態と呼ぶのか』をはっきりさせます。


## 数式より先に見てほしいこと

観測の 2 つのキーが同時に動いていても、内部では 1 つの遷移規則がそれを支えているかもしれません。状態空間モデルは、その隠れた共通原因を置くための道具として読むと自然です。最初は、「見えている項目は2つでも、裏では1つのまとまった動きが支えているかもしれない」と捉えると入りやすくなります。


## この notebook の立ち位置

本格的な推定アルゴリズムまでは踏み込みません。ここでは『隠れ状態を置くと予測と解釈がどう整理されるか』に絞って見ます。読み終わったときに、『見えている値の裏に、もっとなめらかに進む内部メモを置く意味』を説明できれば十分です。


## 観察 1: 状態空間モデルの遷移

状態空間モデルとして、まず内部状態がどう進むかの最小形を定義します。


In [ ]:
z_t = 0.35
a_t = -0.3
A, B = 0.88, 0.22
z_next = A * z_t + B * a_t
print('task = state-space-models')
print('z_next =', round(z_next, 6))

この線形遷移を起点に表現学習へ接続します。ここでは `A`、`B`、`decode` を手書きで置く最小例ですが、実際の状態空間モデルでは観測データからこれらの係数を推定・学習します。まず内部の進み方を仮に決めて、そのあとで外からどう見えるかを作る、という二段階を意識してください。


## 観察 2: 観測予測を作る

次に、潜在状態から観測を復元する写像を作ります。内部状態と見えている量の役割分担をコードで掴みます。ここでは、『内部で持っている作業メモを、外から見える量へ訳し直す』段階を見ています。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、状態の進み方の誤差と、見え方の誤差を分離して調整できます。
つまり、「内部の進み方が悪いのか」「その内部状態を観測へ変える段階が悪いのか」を別々に直せるようになります。



## 計算の対応表

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$

式としては短いですが、意味は『内部メモを1歩進める』と『そのメモを外から見える量へ戻す』の二段階です。


## 観察 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1ステップでは見えない誤差の積み上がりを把握するためです。ロールアウトとは、予測した結果を次の材料としてまた回しながら、先の未来を順にたどることです。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、状態の進み方の安定性や状態表現の情報量不足を疑います。
1歩ずつはそこそこでも、何歩もつなぐと外れるなら、その状態メモが先まで使えるほど整理されていない可能性があります。



## 観察 4: 計画候補を比較する

次に、複数の行動列を比較して、どの計画が望ましいかを評価します。ここで、状態空間モデルが未来予測だけでなく行動選択にもつながることが見えてきます。要するに、「予測のための模型」がそのまま「案を比べるための模型」にもなります。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

計画評価が可能になると、実環境での試行回数を抑えた探索がしやすくなります。
ここでは厳密な報酬設計より、「どの案が目的に近そうかを先に並べ替える」ことが主眼です。



## 観察 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。状態空間モデルも『どこまで信用してよいか』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの条件でモデルが弱いかを特定しやすくなります。
平均だけを見ると隠れてしまう『途中から急に外れる』失敗も見つけやすくなります。



## 読み終えたあとに残したい視点

1. 観測はそのまま内部状態ではない。
2. 状態遷移と観測生成を分けると、どこで誤差が出たか切り分けやすい。
3. 長期予測の安定性は、内部状態の置き方に強く依存する。
